# 🧠 LLM Data Preprocessing Pipeline — Multiple Datasets

This notebook walks through every major step for preprocessing raw text datasets for LLM training:

1. Data Collection & Inventory
2. Data Cleaning (per dataset)
3. Quality Filtering
4. Normalization
5. Tokenization
---

## 📦 Step 0: Install Dependencies

In [1]:
!pip install datasets transformers sentencepiece datasketch langdetect pandas pyarrow tqdm ftfy torch

In [2]:
import os
import re
import json
import hashlib
import unicodedata
import random
import torch
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm
import ftfy

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from datasketch import MinHash, MinHashLSH
from langdetect import detect, LangDetectException
from transformers import AutoTokenizer

print('✅ All libraries imported successfully')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully


---
## 📂 Step 1: Data Collection & Inventory

Define your dataset sources, formats, and metadata. Here we simulate three datasets: web crawl, books, and code.

In [3]:
import os
import json
import random
import numpy as np
import torch
from langdetect import DetectorFactory

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
DetectorFactory.seed = 0  # makes langdetect deterministic

os.environ["DATA_DIR"] = "/workspace/data"

data_dir = os.environ.get("DATA_DIR", "./data")
DATASET_REGISTRY = {}

print("📂 Loading standard JSON files...")

if not os.path.isdir(data_dir):
    print(f"❌ Directory not found: {data_dir}")
else:
    for filename in os.listdir(data_dir):
        if filename.endswith(".json"):
            dataset_name = os.path.splitext(filename)[0]
            file_path = os.path.join(data_dir, filename)
            
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    raw_data = json.load(f)
                
                    if isinstance(raw_data, dict) and 'questions' in raw_data:
                        data_list = raw_data['questions']
                    elif isinstance(raw_data, list):
                        data_list = raw_data
                    else:
                        data_list = [raw_data]

                    DATASET_REGISTRY[dataset_name] = {
                        'data': data_list,
                    }
                except json.JSONDecodeError as e:
                    print(f"⚠️ Could not parse {filename}: {e}")

# Check results
print("\n📋 Dataset Inventory:")
for name, meta in DATASET_REGISTRY.items():
    print(f"  [{name}] — {len(meta['data'])} documents")

📂 Loading standard JSON files...

📋 Dataset Inventory:
  [13B1_golden] — 85 documents
  [13B2_golden] — 85 documents
  [13B3_golden] — 85 documents
  [13B4_golden] — 85 documents
  [test_categorized] — 294 documents
  [training13b] — 5389 documents
  [training_categorized] — 4803 documents


---
## 🧹 Step 2: Data Cleaning

In [4]:
def remove_html_tags(text: str) -> str:
    """Strip HTML tags from text."""
    return re.sub(r'<[^>]+>', '', text)

def fix_encoding(text: str) -> str:
    """Fix mojibake and encoding issues using ftfy."""
    return ftfy.fix_text(text)

def is_low_quality(text: str, min_words: int = 5, max_symbol_ratio: float = 0.3) -> bool:
    """Flag text as low quality based on heuristics."""
    words = text.split()
    if len(words) < min_words:
        return True
    symbols = sum(1 for c in text if not c.isalnum() and not c.isspace())
    if symbols / max(len(text), 1) > max_symbol_ratio:
        return True
    return False

def detect_language(text: str, target_lang: str = "en") -> bool:
    """Return True if text matches the target language."""
    try:
        return detect(text) == target_lang
    except LangDetectException:
        return False

def remove_pii(text: str) -> str:
    """Remove common PII patterns (email, phone, SSN)."""
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[EMAIL]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    return text


def clean_document(doc: dict) -> dict | None:
    """Full cleaning pipeline for a single document."""
    
    # 1. Identify the text field. 
    # In BioASQ 'body' is the question. 
    # If you want to clean snippets, you'll need to join them first.
    text = doc.get("body", "") 
    
    # If 'body' is empty, let's check for 'snippets'
    if not text and "snippets" in doc:
        text = " ".join([s.get("text", "") for s in doc["snippets"]])

    # 2. Run the cleaning steps
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    
    text = remove_pii(text)
    
    # 3. Return the doc with the NEW 'text' field (standardizing it for RAG)
    return {**doc, "text": text}

# Apply cleaning to all datasets
cleaned_datasets = {}
for name, meta in DATASET_REGISTRY.items():
    print(f"🧹 Cleaning dataset: {name} ({len(meta['data'])} documents)")
    cleaned_docs = []
    for doc in tqdm(meta['data'], desc=f"Cleaning {name}"):
        cleaned_doc = clean_document(doc)
        if cleaned_doc:
            cleaned_docs.append(cleaned_doc)
    cleaned_datasets[name] = cleaned_docs
    print(f"✅ Cleaned {len(cleaned_docs)} documents from {name}")

🧹 Cleaning dataset: 13B1_golden (85 documents)


Cleaning 13B1_golden: 100% 85/85 [00:00<00:00, 201.59it/s]


✅ Cleaned 73 documents from 13B1_golden
🧹 Cleaning dataset: 13B2_golden (85 documents)


Cleaning 13B2_golden: 100% 85/85 [00:00<00:00, 354.27it/s]


✅ Cleaned 75 documents from 13B2_golden
🧹 Cleaning dataset: 13B3_golden (85 documents)


...

✅ Cleaned 294 documents from test_categorized
🧹 Cleaning dataset: training13b (5389 documents)


Cleaning training13b: 100% 5389/5389 [00:13<00:00, 407.54it/s]


✅ Cleaned 4804 documents from training13b
🧹 Cleaning dataset: training_categorized (4803 documents)


Cleaning training_categorized: 100% 4803/4803 [00:11<00:00, 431.19it/s]

✅ Cleaned 4803 documents from training_categorized


---
## 🎯 Step 3: Quality Filtering

In [5]:
def avg_word_length(text: str) -> float:
    words = text.split()
    return sum(len(w) for w in words) / max(len(words), 1)

def bullet_density(text: str) -> float:
    lines = text.splitlines()
    bullet_lines = sum(1 for l in lines if l.strip().startswith(('•', '-', '*', '·')))
    return bullet_lines / max(len(lines), 1)

def passes_quality_filter(doc: dict, verbose: bool = False) -> bool:
    text = doc.get("text", "")
    if not text or len(text) < 20:
        if verbose: print(f"FAIL length: {len(text)}")
        return False
    awl = avg_word_length(text)
    if not (2.5 <= awl <= 15):
        if verbose: print(f"FAIL awl: {awl:.2f}")
        return False
    if bullet_density(text) > 0.6:
        if verbose: print(f"FAIL bullets")
        return False
    return True

quality_filtered = {}
for name, docs in cleaned_datasets.items():
    before = len(docs)
    filtered = [doc for doc in docs if passes_quality_filter(doc)]
    quality_filtered[name] = filtered
    print(f"[{name}] quality filter: {before} → {len(filtered)} docs")

[13B1_golden] quality filter: 73 → 73 docs
[13B2_golden] quality filter: 75 → 75 docs
[13B3_golden] quality filter: 76 → 76 docs
[13B4_golden] quality filter: 70 → 70 docs
[test_categorized] quality filter: 294 → 294 docs
[training13b] quality filter: 4804 → 4803 docs
[training_categorized] quality filter: 4803 → 4803 docs


---
## ✏️ Step 4: Text Normalization

In [6]:
def normalize_text(text: str, form: str = "NFC") -> str:
    """Unicode normalization + whitespace cleanup."""
    # Unicode normalization
    text = unicodedata.normalize(form, text)
    # Normalize whitespace
    text = re.sub(r'\r\n|\r', '\n', text)         # normalize line endings
    text = re.sub(r'\t', ' ', text)               # tabs → spaces
    text = re.sub(r' {2,}', ' ', text)            # collapse multiple spaces
    text = re.sub(r'\n{3,}', '\n\n', text)        # collapse excess blank lines
    text = text.strip()
    return text

normalized_datasets = {}
for name, docs in quality_filtered.items():
    normalized = [{**doc, "text": normalize_text(doc["text"])} for doc in docs]
    normalized_datasets[name] = normalized

training_raw_data = normalized_datasets["training13b"]
test_raw_data = normalized_datasets["13B1_golden"] + normalized_datasets["13B2_golden"] + normalized_datasets["13B3_golden"] + normalized_datasets["13B4_golden"]

print("✅ Normalization complete")

# Preview a sample
for name, docs in normalized_datasets.items():
    if docs:
        print(f"\n[{name}] sample: {docs[0]['text'][:120]}..." if len(docs[0]['text']) > 120 else f"\n[{name}] sample: {docs[0]['text']}")

✅ Normalization complete

[13B1_golden] sample: What proportion of colorectal cancer cases are not assignable to any of the consensus molecular subtype (CMS) groups?

[13B2_golden] sample: Which ensemble machine-learning framework has been developed harnessing UK biobank data?

[13B3_golden] sample: How many primary genetic associations were identified through pQTL mapping within the Pharma Proteomics Project?

[13B4_golden] sample: Should Zotiraciclib be used for glioblastoma?

[test_categorized] sample: What proportion of colorectal cancer cases are not assignable to any of the consensus molecular subtype (CMS) groups?

[training13b] sample: Is Hirschsprung disease a mendelian or a multifactorial disorder?

[training_categorized] sample: Is Hirschsprung disease a mendelian or a multifactorial disorder?


---
## 🔤 Step 5: Categories

In [7]:
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from umap import UMAP
from hdbscan import HDBSCAN

texts = [doc['text'][:500] for doc in training_raw_data]

# Dynamic parameters based on dataset size
n_docs = len(texts)
min_cluster_size = max(5, int(n_docs * 0.01))
n_neighbors = min(15, int(n_docs * 0.05))

print(f'Docs: {n_docs}')
print(f'min_cluster_size: {min_cluster_size}')
print(f'n_neighbors: {n_neighbors}')

representation_model = MaximalMarginalRelevance(diversity=0.3)

topic_model = BERTopic(
    umap_model=UMAP(
        random_state=SEED,
        n_neighbors=n_neighbors,
        n_components=5,
        min_dist=0.0
    ),
    hdbscan_model=HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=1,
        prediction_data=True
    ),
    representation_model=representation_model,
    language='english',
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(texts)

# Assign back to docs
for doc, topic in zip(training_raw_data, topics):
    if topic == -1:
        doc['category'] = 'Uncategorized'
    else:
        keywords = topic_model.get_topic(topic)
        top3 = ' & '.join([w.title() for w, _ in keywords[:3]])
        doc['category'] = top3
    doc['topic_id'] = topic

topic_info = topic_model.get_topic_info()
print(topic_info[['Topic', 'Name', 'Count']].head(20))

from collections import Counter
cats = Counter(doc['category'] for doc in training_raw_data)
print(f'\nTotal categories: {len(cats)}')
print(f'Uncategorized: {cats.get("Uncategorized", 0)}')

2026-03-07 04:21:30,990 - BERTopic - Embedding - Transforming documents to embeddings.


Docs: 4803
min_cluster_size: 48
n_neighbors: 15


Loading weights: 100% 103/103 [00:00<00:00, 643.08it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100% 151/151 [00:01<00:00, 104.54it/s]
2026-03-07 04:21:36,899 - BERTopic - Embedding - Completed ✓
2026-03-07 04:21:36,900 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-07 04:21:58,522 - BERTopic - Dimensionality - Completed ✓
2026-03-07 04:21:58,524 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-07 04:21:58,677 - BERTopic - Cluster - Completed ✓
2026-03-07 04:21:58,681 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-07 04:21:59,100 - BERTopi

    Topic                                          Name  Count
0      -1        -1_in_receptor_transcription_treatment    991
1       0          0_syndrome_mutations_symptoms_caused    703
2       1            1_protein_function_proteins_kinase    572
3       2   2_package_algorithm_rbioconductor_conserved    233
4       3       3_yeast_enhancers_chromatin_methylation    183
5       4         4_treatment_diseases_arthritis_crohns    179
6       5      5_thyroid_hormone_cardiomyopathy_calcium    167
7       6             6_covid19_infection_viruses_fever    160
8       7                7_approved_fda_clinical_trials    122
9       8          8_mechanism_mode_pitolisant_elagolix    122
10      9                   9_genes_alu_elegans_genomic    105
11     10             10_bacteria_gut_clostridium_genus    102
12     11                 11_score_pain_depression_used    102
13     12              12_amyotrophic_sclerosis_als_tau    100
14     13              13_juice_enzymes_inhibitor_nadph

In [8]:
# See all topic names with keywords
print("\n📋 All discovered topics:")
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    keywords = topic_model.get_topic(topic_id)
    top3 = ' & '.join([w.title() for w, _ in keywords[:3]])
    count = sum(1 for t in topics if t == topic_id)
    print(f"  Topic {topic_id:>3}: {top3:<50} ({count} docs)")


📋 All discovered topics:
  Topic   0: Syndrome & Mutations & Symptoms                    (703 docs)
  Topic   1: Protein & Function & Proteins                      (572 docs)
  Topic   2: Package & Algorithm & Rbioconductor                (233 docs)
  Topic   3: Yeast & Enhancers & Chromatin                      (183 docs)
  Topic   4: Treatment & Diseases & Arthritis                   (179 docs)
  Topic   5: Thyroid & Hormone & Cardiomyopathy                 (167 docs)
  Topic   6: Covid19 & Infection & Viruses                      (160 docs)
  Topic   7: Approved & Fda & Clinical                          (122 docs)
  Topic   8: Mechanism & Mode & Pitolisant                      (122 docs)
  Topic   9: Genes & Alu & Elegans                              (105 docs)
  Topic  10: Bacteria & Gut & Clostridium                       (102 docs)
  Topic  11: Score & Pain & Depression                          (102 docs)
  Topic  12: Amyotrophic & Sclerosis & Als                      (100 docs)

In [9]:
from transformers import pipeline

print('Loading classifier...')
classifier = pipeline(
    'zero-shot-classification',
    model='cross-encoder/nli-deberta-v3-small',
    device=0 if torch.cuda.is_available() else -1
)
print('Classifier ready!')

Loading classifier...


Loading weights: 100% 106/106 [00:00<00:00, 713.39it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Classifier ready!


In [10]:
from tqdm import tqdm
from collections import Counter

# Only classify docs that are still Uncategorized
uncategorized_docs = [doc for doc in training_raw_data if doc['category'] == 'Uncategorized']
print(f'Classifying {len(uncategorized_docs)} uncategorized docs...')

# Use categories already discovered from BERTopic
DISCOVERED_CATEGORIES = list(set(
    doc['category'] for doc in training_raw_data 
    if doc['category'] != 'Uncategorized'
))
print(f'Against {len(DISCOVERED_CATEGORIES)} categories')

classified = 0
for doc in tqdm(uncategorized_docs, desc='Classifying'):
    result = classifier(doc['text'][:512], DISCOVERED_CATEGORIES)
    if result['scores'][0] > 0.3:
        doc['category'] = result['labels'][0]
        doc['confidence'] = result['scores'][0]
        classified += 1
    else:
        doc['category'] = 'Other'

print(f'\n✅ Classified: {classified}')
print(f'❌ Fell to Other: {len(uncategorized_docs) - classified}')

# Check final distribution
cats = Counter(doc['category'] for doc in training_raw_data)
print(f'\nUncategorized remaining: {cats.get("Uncategorized", 0)}')

Classifying 991 uncategorized docs...
Against 26 categories


Classifying:   1% 10/991 [00:02<02:39,  6.16it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Classifying: 100% 991/991 [02:43<00:00,  6.07it/s]


✅ Classified: 3
❌ Fell to Other: 988

Uncategorized remaining: 0


In [11]:
# ── Cell 10: Save Categorized Data ─────────────────────────────────
import json

# Save training data
train_output = '/workspace/data/training_categorized.json'
with open(train_output, 'w', encoding='utf-8') as f:
    json.dump(training_raw_data, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(training_raw_data)} training docs to {train_output}')

# Save test data
test_output = '/workspace/data/test_categorized.json'
with open(test_output, 'w', encoding='utf-8') as f:
    json.dump(test_raw_data, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(test_raw_data)} test docs to {test_output}')

# Verify
with open(train_output, 'r', encoding='utf-8') as f:
    verify = json.load(f)
print(f'\n✅ Verified: {len(verify)} docs loaded back')
print(f'   Fields: {list(verify[0].keys())}')
print(f'   Sample category: {verify[0]["category"]}')
print(f'   Sample text: {verify[0]["text"][:100]}...')

✅ Saved 4803 training docs to /workspace/data/training_categorized.json
✅ Saved 294 test docs to /workspace/data/test_categorized.json

✅ Verified: 4803 docs loaded back
   Fields: ['body', 'documents', 'ideal_answer', 'concepts', 'type', 'id', 'snippets', 'text', 'category', 'topic_id']
   Sample category: Syndrome & Mutations & Symptoms
   Sample text: Is Hirschsprung disease a mendelian or a multifactorial disorder?...


In [1]:
# Run this in a NEW cell while the other is running
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f'Memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB used')

True
NVIDIA GeForce RTX 5070 Ti Laptop GPU
Memory: 0.00 GB used


In [1]:
import pandas as pd

df = pd.read_csv('/workspace/data/parsed_docs/training_final.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (4803, 7)


,question,text,category,topic_id,confidence,n_sources,source_urls
0,Is Hirschsprung disease a mendelian or a multi...,1. Antimicrob Agents Chemother. 2018 Sep 24;62...,Infectious Disease,5,0.6771,16,"http://www.ncbi.nlm.nih.gov/pubmed/15829955, h..."
1,List signaling molecules (ligands) that intera...,1. Eur J Pharmacol. 2009 Mar 1;605(1-3):109-13...,Pharmacology & Drugs,2,0.3402,15,"http://www.ncbi.nlm.nih.gov/pubmed/24323361, h..."
2,Is the protein Papilin secreted?,1. J Neurol Sci. 2009 Aug 15;283(1-2):109-15. ...,Rare Diseases,8,0.1489,8,"http://www.ncbi.nlm.nih.gov/pubmed/21784067, h..."
3,Are long non coding RNAs spliced?,1. Eur J Radiol. 2015 Apr;84(4):709-20. doi: 1...,Diagnostics & Imaging,10,0.2847,6,"http://www.ncbi.nlm.nih.gov/pubmed/22955988, h..."
4,Is RANKL secreted from the cells?,1. Int J Neurosci. 2013 May;123(5):324-8. doi:...,Neurology & Brain,3,0.1222,11,"http://www.ncbi.nlm.nih.gov/pubmed/24267510, h..."


In [2]:
sample_url = list(abstract_cache.keys())[0]
print(abstract_cache[sample_url])

NameError: name 'abstract_cache' is not defined

In [3]:
sample_url = list(abstract_cache.keys())[0]
print(repr(abstract_cache[sample_url][:1000]))

NameError: name 'abstract_cache' is not defined